# Notebook 1: Data Visualisation And Dataset Split

This notebook checks annotation format, visualises YOLO and Faster R-CNN boxes, and creates the shared train/val/test split used by both models.

In [1]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw


def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "medical_detection").exists():
            return candidate

    kaggle_candidate = Path("/kaggle/working/model-comparison-for-lesion-detection")
    if (kaggle_candidate / "medical_detection").exists():
        return kaggle_candidate

    raise FileNotFoundError("Could not locate a project root containing the medical_detection package.")


PROJECT_ROOT = resolve_project_root(Path.cwd())
DATASET_ROOT = Path("/kaggle/input/datasets/capsuleyolo/kyucapsule") if Path("/kaggle/input/datasets/capsuleyolo/kyucapsule").exists() else PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from medical_detection import ProjectPaths

paths = ProjectPaths(project_root=PROJECT_ROOT, dataset_root=DATASET_ROOT)
random_seed = 42
random.seed(random_seed)

print(f"PROJECT_ROOT: {paths.project_root}")
print(f"DATASET_ROOT: {paths.dataset_root}")
print(f"image_dir: {paths.image_dir}")
print(f"label_dir: {paths.label_dir}")

PROJECT_ROOT: c:\Users\Domagoj\Downloads\malter
DATASET_ROOT: c:\Users\Domagoj\Downloads\malter
image_dir: c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_images\SEE_AI_project_all_images
label_dir: c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_txt\SEE_AI_project_all_txt


In [2]:
from medical_detection import build_dataset_index

dataset_index = build_dataset_index(paths.image_dir, paths.label_dir, paths.csv_path, group_size=25)
image_records = list(dataset_index.records)
image_paths = [record.image_path for record in image_records]
annotation_paths = [record.label_path for record in image_records]

print(f"Found {len(image_paths)} images")
print(f"Found {len(annotation_paths)} annotations")
print(f"Positive images: {len(dataset_index.positive_records)}")
print(f"Negative images: {len(dataset_index.negative_records)}")
print("First 5 images:")
for record in image_records[:5]:
    print(" ", record.image_path)

Found 18481 images
Found 18481 annotations
Positive images: 12290
Negative images: 6191
First 5 images:
  c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_images\SEE_AI_project_all_images\image00001.jpg
  c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_images\SEE_AI_project_all_images\image00002.jpg
  c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_images\SEE_AI_project_all_images\image00003.jpg
  c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_images\SEE_AI_project_all_images\image00004.jpg
  c:\Users\Domagoj\Downloads\malter\SEE_AI_project_all_images\SEE_AI_project_all_images\image00005.jpg


In [3]:
class_names = list(dataset_index.class_names)
nc = len(class_names)
class_mapping = {index: name for index, name in enumerate(class_names)}

print(f"nc = {nc}")
print(class_names)

nc = 11
['angiodysplasia', 'erosion', 'stenosis', 'lymphangiectasia', 'lymph follicle', 'SMT', 'polyp-like', 'bleeding', 'erythema', 'foreign body', 'vein']


In [4]:
from medical_detection import convert_yolo_to_xyxy


def annotations_for_image(image_path):
    return dataset_index.by_filename[Path(image_path).name].annotations


def draw_yolo_boxes(image, annotations):
    draw = ImageDraw.Draw(image)
    img_width, img_height = image.size
    for ann in annotations:
        box = convert_yolo_to_xyxy(ann.x_center, ann.y_center, ann.width, ann.height, img_width, img_height)
        draw.rectangle([box.xmin, box.ymin, box.xmax, box.ymax], outline="red", width=3)
        draw.text((box.xmin, max(0, box.ymin - 15)), class_names[ann.class_id], fill="red")
    return image


def draw_faster_rcnn_boxes(image, annotations):
    draw = ImageDraw.Draw(image)
    for ann in annotations:
        draw.rectangle([ann["xmin"], ann["ymin"], ann["xmax"], ann["ymax"]], outline="green", width=3)
        draw.text((ann["xmin"], max(0, ann["ymin"] - 15)), ann["class_name"], fill="green")
    return image

In [5]:
from medical_detection import (
    box_level_class_distribution,
    downsample_negative_records,
    grouped_split,
    image_level_class_distribution,
    write_split_files,
)

splits = grouped_split(dataset_index, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=random_seed)

train_records = splits["train"]
val_records = splits["val"]
test_records = splits["test"]

train_background_target = 0.15
train_records_bg15 = downsample_negative_records(
    train_records,
    target_negative_ratio=train_background_target,
    seed=random_seed,
 )
splits_bg15 = {"train": train_records_bg15, "val": val_records, "test": test_records}

print(f"train={len(train_records)}, val={len(val_records)}, test={len(test_records)}")
print(f"Seed used: {random_seed}")


def negative_count(records):
    return sum(1 for record in records if not record.has_annotations)


def negative_ratio(records):
    return (negative_count(records) / len(records)) if records else 0.0


def print_split_summary(split_name, records):
    image_counts = image_level_class_distribution(records, nc)
    box_counts = box_level_class_distribution(records, nc)
    split_negative_count = negative_count(records)
    print(f"\n{split_name}: {len(records)} images")
    print(f"Negative images: {split_negative_count} ({negative_ratio(records):.2%})")
    for class_id in range(nc):
        image_percentage = (image_counts[class_id] / len(records) * 100.0) if records else 0.0
        print(
            f"{class_id:2d} {class_names[class_id]:20s}: "
            f"{image_counts[class_id]:5d} images ({image_percentage:6.2f}%), "
            f"{box_counts[class_id]:5d} boxes"
        )


print_split_summary("train", train_records)
print_split_summary("val", val_records)
print_split_summary("test", test_records)

print(f"\ntrain_bg15 target negative ratio: {train_background_target:.0%}")
print_split_summary("train_bg15", train_records_bg15)

train=12931, val=3700, test=1850
Seed used: 42

train: 12931 images
Negative images: 4530 (35.03%)
 0 angiodysplasia      :   568 images (  4.39%),   643 boxes
 1 erosion             :  2953 images ( 22.84%),  3916 boxes
 2 stenosis            :   325 images (  2.51%),   328 boxes
 3 lymphangiectasia    :   423 images (  3.27%),   453 boxes
 4 lymph follicle      :  1218 images (  9.42%),  4606 boxes
 5 SMT                 :   370 images (  2.86%),   380 boxes
 6 polyp-like          :  1403 images ( 10.85%),  2087 boxes
 7 bleeding            :   605 images (  4.68%),   648 boxes
 8 erythema            :   646 images (  5.00%),   711 boxes
 9 foreign body        :   406 images (  3.14%),  1139 boxes
10 vein                :   459 images (  3.55%),   635 boxes

val: 3700 images
Negative images: 1156 (31.24%)
 0 angiodysplasia      :   188 images (  5.08%),   210 boxes
 1 erosion             :   962 images ( 26.00%),  1315 boxes
 2 stenosis            :    59 images (  1.59%),    59 boxe

In [6]:
import json

write_split_files(paths.splits_dir, splits)
write_split_files(paths.splits_dir, splits, absolute_paths=True, suffix="_absolute")
write_split_files(paths.splits_dir, splits_bg15, suffix="_bg15")
write_split_files(paths.splits_dir, splits_bg15, absolute_paths=True, suffix="_bg15_absolute")

data_yaml_path = paths.splits_dir / "data.yaml"
with data_yaml_path.open("w", encoding="utf-8") as handle:
    handle.write(f"train: {(paths.splits_dir / 'train_absolute.txt').as_posix()}\n")
    handle.write(f"val: {(paths.splits_dir / 'val_absolute.txt').as_posix()}\n")
    handle.write(f"test: {(paths.splits_dir / 'test_absolute.txt').as_posix()}\n")
    handle.write(f"nc: {nc}\n")
    handle.write(f"names: {class_names}\n")

data_bg15_yaml_path = paths.splits_dir / "data_bg15.yaml"
with data_bg15_yaml_path.open("w", encoding="utf-8") as handle:
    handle.write(f"train: {(paths.splits_dir / 'train_bg15_absolute.txt').as_posix()}\n")
    handle.write(f"val: {(paths.splits_dir / 'val_bg15_absolute.txt').as_posix()}\n")
    handle.write(f"test: {(paths.splits_dir / 'test_bg15_absolute.txt').as_posix()}\n")
    handle.write(f"nc: {nc}\n")
    handle.write(f"names: {class_names}\n")

split_manifest = {
    "project_root": str(paths.project_root),
    "dataset_root": str(paths.dataset_root),
    "image_dir": str(paths.image_dir),
    "label_dir": str(paths.label_dir),
    "group_size": 25,
    "random_seed": random_seed,
    "class_names": class_names,
    "counts": {name: len(records) for name, records in splits.items()},
    "negative_counts": {name: negative_count(records) for name, records in splits.items()},
    "experiments": {
        "bg15": {
            "target_train_negative_ratio": train_background_target,
            "counts": {name: len(records) for name, records in splits_bg15.items()},
            "negative_counts": {name: negative_count(records) for name, records in splits_bg15.items()},
            "split_suffix": "_bg15",
        }
    },
}
manifest_path = paths.splits_dir / "split_manifest.json"
manifest_path.write_text(json.dumps(split_manifest, indent=2), encoding="utf-8")

print(f"baseline data.yaml saved to {data_yaml_path}")
print(f"bg15 data.yaml saved to {data_bg15_yaml_path}")
print(f"split manifest saved to {manifest_path}")

baseline data.yaml saved to c:\Users\Domagoj\Downloads\malter\dataset_splits\data.yaml
bg15 data.yaml saved to c:\Users\Domagoj\Downloads\malter\dataset_splits\data_bg15.yaml
split manifest saved to c:\Users\Domagoj\Downloads\malter\dataset_splits\split_manifest.json
